# Logical Channel Validation

Sanity checks plus two independent cross-checks of `LogicalChannel`: the brute-force fault-path enumeration (`brute_force_channel.py`) and the Mathematica-derived closed forms (`theory.py`).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from ClassicalCode import ClassicalCode
from LogicalChannel import LogicalChannel
from measures import logical_basis_state
from error_models import (perfect_physical_error, perfect_syndrome,
                          iid_bitflip_error, iid_syndrome)
from random_codes import sample_H
from brute_force_channel import brute_logical_channel
from theory import (theory_result_perfect_syndrome_t3,
                    theory_result_perfect_physical_t3)

# Logical Channel Sanity Check

In [2]:
# No Error Must be Identity
n, k = 3, 1
q = 0.1
T_max = 50

c = ClassicalCode(sample_H(n,k))

channel = LogicalChannel(c, T_max, q, p_error=perfect_physical_error, syndrome=perfect_syndrome(c))

a = 0.3

rho_in = a * logical_basis_state(c, 0) + (1-a) * logical_basis_state(c, 1)

rho_out = channel(T_max, rho_in)

print(rho_in == rho_out)


[[ True  True]
 [ True  True]]


In [3]:
# Simple Case


In [4]:
# ---- exact cross-check: full enumeration must equal LogicalChannel ----
code_v = ClassicalCode(np.array([[1, 1, 0], [0, 1, 1]]))
for T in (2, 3):
    A_full = brute_logical_channel(code_v, T, 0.2, max_faults=None)   # all faults, normalized (=raw)
    A_lc   = LogicalChannel(code_v, T, 0.2).stochastic_matrix(T)
    print(f'T={T}: full-enumeration vs LogicalChannel maxdiff = {np.max(np.abs(A_full - A_lc)):.2e}')

T=2: full-enumeration vs LogicalChannel maxdiff = 3.89e-16
T=3: full-enumeration vs LogicalChannel maxdiff = 1.20e-13


In [5]:
# ---- the <=2-fault channel, conditioned on <=2 faults (columns = 1) ----
T, q = 8, 0.05
A2 = brute_logical_channel(code_v, T, q, max_faults=2, normalize = False)               # normalize=True (default)
print(f'\n<=2-fault logical channel, conditioned (T={T}, q={q}):')
print(np.round(A2, 6))
print('column sums:', np.round(A2.sum(0), 6))

B = (13*T -6) * (q**2)
print(B)

print(np.array([[1-B,B],[B,1-B]]))


<=2-fault logical channel, conditioned (T=8, q=0.05):
[[0.641849 0.034887]
 [0.034887 0.641849]]
column sums: [0.676736 0.676736]
0.24500000000000005
[[0.755 0.245]
 [0.245 0.755]]


# Logical Channel Result

In [6]:
#code = ClassicalCode(np.array([[1,1,0,0,0],[0,1,1,0,0],[0,0,1,1,0],[0,0,0,1,1]]))

code = ClassicalCode(np.array([[1,1,0],[0,1,1]]))

## Compare with Mathematica

spceifically for time step T = 3, and q = 0.1, can derive the mathematiacl formula for the logical channel in mathematica. For other time steps, would have to change the functions "theory result ..." as well as the exact numbers. Wanted to check if the implemented channel works correctly. seems to do so.

### no sndrome error, iid physical error, Time step = 3

In [7]:
q = 0.1
Tmax = 3

channel =  LogicalChannel(code, Tmax, q, syndrome=perfect_syndrome(code), p_error=iid_bitflip_error)
a = 0.01
rho0= a * logical_basis_state(code, 0) + (1-a) * logical_basis_state(code, 1)

dim_L = 2 ** code.k

print(channel(Tmax, rho0))

print(theory_result_perfect_syndrome_t3(q,a))

[[0.08779613+0.j 0.        +0.j]
 [0.        +0.j 0.91220387+0.j]]
[[0.08779613 0.        ]
 [0.         0.91220387]]


### no physical error, iid syndrome error, T = 3 step

In [8]:
#see the resulting state when independent noise in physical and no error in syndrome. for T = 3 steps. compare it with the mathematica result.
q = 0.1
Tmax = 3

channel =  LogicalChannel(code, Tmax, q, p_error=perfect_physical_error, syndrome=iid_syndrome(code,q))
a = 0.3
rho0= a * logical_basis_state(code, 0) + (1-a) * logical_basis_state(code, 1)

dim_L = 2 ** code.k

print(channel(Tmax, rho0))

print(theory_result_perfect_physical_t3(q,a))

[[0.3141408+0.j 0.       +0.j]
 [0.       +0.j 0.6858592+0.j]]
[[0.3141408 0.       ]
 [0.        0.6858592]]


### iid syndrome error, iid physical error time step = 3

Can check and compare with the results with Mathematica file for other time steps as well.

In [9]:
q = 0.1
Tmax = 3

channel = LogicalChannel(code, Tmax, q)
a = 0.3
rho0 = a * logical_basis_state(code,0) + (1-a) * logical_basis_state(code,1)

dim_L = 2 ** code.k

print(channel(Tmax, rho0))

print(np.array([[(449661961*(1-a) + 1991744289 * a)/2441406250, 0],[0, (1991744289 * (1-a) + 449661961 * a)/2441406250]]))


[[0.37367262+0.j 0.        +0.j]
 [0.        +0.j 0.62632738+0.j]]
[[0.37367262 0.        ]
 [0.         0.62632738]]
